# Feature Engineering — Psychotherapy Dropout Prediction

This notebook builds **session-level clinical features** from synthetic psychotherapy data: PHQ-9 change rate over time, whether gaps between appointments are widening (a dropout warning sign), and longest attendance streaks. We track shapes after each step, visualize distributions, relate new features to **dropout**, and save a processed table for modeling.

## Imports and project path

We load **pandas**, **numpy**, **matplotlib**, and **seaborn** for tables, random expansion of cohort data to sessions, and plots. Feature-engineering functions live under `src/`; adding the project root to `sys.path` lets us import them in the notebook the same way as in the rest of the project.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Project root (folder that contains `src/`) on Python path
for _path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))):
    if os.path.isdir(os.path.join(_path, "src")) and _path not in sys.path:
        sys.path.insert(0, _path)
        break

from src.data_loader import generate_synthetic_data
from src.feature_engineering import (
    compute_attendance_streak,
    compute_phq9_change_rate,
    compute_session_gap_pattern,
    run_all_features,
)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## Load synthetic cohort and build session-level data

`generate_synthetic_data()` returns **one row per patient**. The feature functions expect **longitudinal** rows (multiple sessions per patient) with `phq9_score`, `gap_between_sessions_days`, and `attended` per session. We **expand** each patient into a session sequence (at least three sessions so gap patterns are meaningful), then print the shape. This mirrors how real EHR data would list one row per visit.

In [ ]:
def expand_to_longitudinal(df_flat: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """One row per patient -> one row per session (synthetic session history)."""
    rng = np.random.default_rng(seed)
    rows = []
    for _, r in df_flat.iterrows():
        pid = int(r["patient_id"])
        n_sess = max(3, int(r["session_number"]))
        n_sess = min(n_sess, 20)
        base_phq9 = float(r["phq9_score"])
        for s in range(1, n_sess + 1):
            t = (s - 1) / max(1, n_sess - 1)
            phq9 = int(np.clip(base_phq9 + (rng.random() - 0.5) * 8 * t, 0, 27))
            gap = int(rng.integers(3, 61))
            attended = 1 if rng.random() < float(r["attendance_consistency"]) else 0
            rows.append(
                {
                    "patient_id": pid,
                    "session_number": s,
                    "phq9_score": phq9,
                    "gap_between_sessions_days": gap,
                    "attended": attended,
                    "dropout": r["dropout"],
                    "mood_rating": r["mood_rating"],
                    "age": r["age"],
                    "session_frequency_per_month": r["session_frequency_per_month"],
                }
            )
    return pd.DataFrame(rows)


df_flat = generate_synthetic_data(n_patients=500, seed=42)
print("Flat cohort shape:", df_flat.shape)

df = expand_to_longitudinal(df_flat, seed=42)
print("Longitudinal session-level shape:", df.shape)

## PHQ-9 change rate (per session)

**Clinical meaning:** PHQ-9 tracks depression severity; the **rate of change across sessions** indicates whether symptoms are improving (negative slope) or worsening. We run `compute_phq9_change_rate`, compare column names before/after, print shape, and plot the distribution of the new feature.

In [ ]:
cols_before = list(df.columns)
df_phq9 = compute_phq9_change_rate(df)
cols_after = list(df_phq9.columns)
print("Columns before:", cols_before)
print("Columns after:", cols_after)
print("Shape after PHQ-9 feature:", df_phq9.shape)

plt.figure(figsize=(8, 5))
sns.histplot(df_phq9["phq9_change_rate"], kde=True, color="#6366f1")
plt.title("Distribution of PHQ-9 change rate (per session)")
plt.xlabel("PHQ-9 change rate (negative = improving)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Session gap pattern (widening gaps)

**Clinical meaning:** **Increasing gaps** between sessions can signal disengagement and higher dropout risk. We run `compute_session_gap_pattern`, print shape, count how many **unique patients** have `gap_increasing == 1`, and show a bar chart of counts (0 vs 1).

In [ ]:
df_gap = compute_session_gap_pattern(df_phq9)
print("Shape after gap pattern feature:", df_gap.shape)

patient_flags = df_gap.drop_duplicates("patient_id")[["patient_id", "gap_increasing"]]
n_increasing = (patient_flags["gap_increasing"] == 1).sum()
n_total = len(patient_flags)
print(f"Patients with increasing gaps: {n_increasing} / {n_total}")

counts = patient_flags["gap_increasing"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
sns.barplot(x=counts.index.astype(str), y=counts.values, palette=["#86efac", "#fca5a5"])
plt.title("Patients by gap pattern (0 = not increasing, 1 = increasing)")
plt.xlabel("gap_increasing")
plt.ylabel("Number of patients")
plt.tight_layout()
plt.show()

## Attendance streak (longest consecutive attended sessions)

**Clinical meaning:** **Consistent attendance** is associated with better outcomes and lower dropout. `compute_attendance_streak` adds `max_attendance_streak` per patient (repeated on each row). We print shape and plot the distribution of streak lengths **one row per patient** to avoid double-counting.

In [ ]:
df_streak = compute_attendance_streak(df_gap)
print("Shape after attendance streak feature:", df_streak.shape)

streak_per_patient = df_streak.drop_duplicates("patient_id")["max_attendance_streak"]
plt.figure(figsize=(8, 5))
sns.histplot(streak_per_patient, kde=False, bins=range(0, int(streak_per_patient.max()) + 2), color="#0ea5e9")
plt.title("Distribution of longest attendance streak (sessions)")
plt.xlabel("max_attendance_streak")
plt.ylabel("Number of patients")
plt.tight_layout()
plt.show()

## Full pipeline: `run_all_features`

**Clinical meaning:** This runs all three steps in order on the **same** session-level table, producing the final engineered columns used for modeling. We print shape and column list after the pipeline.

In [ ]:
df_final = run_all_features(df)
print("Shape after run_all_features:", df_final.shape)
print("All columns:", list(df_final.columns))

## New features vs. dropout

**Clinical meaning:** We summarize **linear association** (Pearson correlation) between each engineered feature and the binary **dropout** label. Stronger |correlation| suggests the feature may help the model separate patients who leave treatment early. This is exploratory—not causal.

In [ ]:
new_features = ["phq9_change_rate", "gap_increasing", "max_attendance_streak"]
corr_with_dropout = df_final[new_features + ["dropout"]].corr(numeric_only=True)["dropout"].drop("dropout")

plt.figure(figsize=(8, 4))
sns.barplot(x=corr_with_dropout.index, y=corr_with_dropout.values, palette="viridis")
plt.title("Correlation of engineered features with dropout")
plt.xlabel("Feature")
plt.ylabel("Correlation with dropout")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

print("Correlation with dropout:\n", corr_with_dropout)

## Save processed dataset

**Clinical meaning:** Persisting the **session-level table with engineered features** lets you reload the same artifact for training notebooks, DVC, or reproducible experiments without re-running the pipeline.

In [ ]:
# Project root: use parent folder when running from notebooks/
if os.path.basename(os.path.normpath(os.getcwd())).lower() == "notebooks":
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
else:
    project_root = os.getcwd()

out_dir = os.path.join(project_root, "data", "processed")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "featured_data.csv")
df_final.to_csv(out_path, index=False)
print("Saved:", out_path, "| shape:", df_final.shape)

## Key observations (fill in after you run the notebook)

- Which engineered feature has the **strongest correlation** (positive or negative) with `dropout`?
- Does **PHQ-9 change rate** tend toward worsening (positive) for patients who drop out?
- Do **increasing gaps** (`gap_increasing`) align with higher dropout in this synthetic cohort?
- Are **longer attendance streaks** associated with lower dropout risk?

_Record brief bullet notes here for your report; compare with clinical expectations (engagement, symptoms, timing)._ 